# Richmond, VA — Land Value Tax Model

**Model type**: Split-rate 4:1 (land taxed at 4× the improvement rate)  
**Levy scope**: City levy only (Richmond is an independent Virginia city; the $1.20/$100 rate covers all city operations including schools)  
**Tax rate**: $1.20 per $100 of assessed value = 12.0 mills  
**Assessment basis**: 100% market value (Virginia law requires assessment at fair market value; no assessment ratio)  
**Exemptions**: Full exemptions preserved (TaxExemptCode IS NOT NULL → excluded from reform base)  
**Data source**: City of Richmond Assessor of Real Estate GIS FeatureServer (ArcGIS Online, org: richmondvagis)  
**Revenue target**: ~$494M FY2025 (per City of Richmond FY2025 budget projections at $1.20/$100 rate)  

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `LandValue` | Full market value from CAMA system |
| Improvement value | `DwellingValue` | Buildings/structures; null = no improvement |
| Total value | `TotalValue` | LandValue + DwellingValue |
| Use/class code | `PropertyClassID` | City assessor class code (3-digit) |
| Class description | `PropertyClass` | Human-readable class name |
| Exemption flag | `TaxExemptCode` | Non-null = fully exempt from local real estate tax |
| Parcel ID | `PIN` | Property Identification Number |

**Assessment ratio**: None (100% market value per Virginia Code § 58.1-3201)  
**Millage source**: Adopted city rate $1.20/$100 = 12.0 mills; held steady since ~2008; confirmed Nov 2024 council vote  
**Note**: Richmond is a Virginia independent city — no county levy; city rate is the entire real estate tax burden.

## Section 1 — Imports and Setup

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'richmond'
STATE_FIPS = '51'
COUNTY_FIPS = '760'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Richmond adopted real estate tax rate: $1.20 per $100 assessed value
# Virginia assesses at 100% market value — no assessment ratio
MILLAGE = 12.0  # mills per $1,000 AV
OFFICIAL_REVENUE = 494_000_000  # FY2025 estimate at $1.20/$100 rate

print(f"City: {CITY_NAME}")
print(f"FIPS: {STATE_FIPS}{COUNTY_FIPS}")
print(f"Millage: {MILLAGE} mills  (${MILLAGE/10:.2f} per $100 AV)")

City: richmond
FIPS: 51760
Millage: 12.0 mills  ($1.20 per $100 AV)


## Section 2 — Fetch / Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    # City of Richmond Assessor of Real Estate — ArcGIS Online FeatureServer
    # Item ID: fbfce2aab2a44c05bc0abc2d6ea7e54a (org: richmondvagis / k3vhq11XkBNeeOfM)
    # get_feature_data_with_geometry builds: {base_url}/{dataset_name}/FeatureServer/{layer_id}/query
    gdf = get_feature_data_with_geometry(
        dataset_name='Parcels',
        base_url='https://services1.arcgis.com/k3vhq11XkBNeeOfM/arcgis/rest/services',
        layer_id=0,
        paginate=True,
    )
    # Richmond is an independent city — all parcels in this service are within the city
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(f"Columns: {list(gdf.columns)}")
print(f"CRS: {gdf.crs}")

Layer metadata CRS WKID: 2284
Total records in Parcels: 76879


Query response spatialReference: {'wkid': 4326, 'latestWkid': 4326}
Fetched records 0 to 1000


Fetched records 1000 to 2000


Fetched records 2000 to 3000


Fetched records 3000 to 4000


...

Fetched records 74000 to 75000


Fetched records 75000 to 76000


Fetched records 76000 to 76879


Total bounds: [-77.601175  37.447529 -77.385515  37.602817]


Downloaded and cached 76,879 parcels
Columns: ['ParcelID', 'PIN', 'CountOfPIN', 'OwnerName', 'AsrLocationBldgNo', 'MailAddress', 'MailCity', 'MailState', 'MailZip', 'AssessmentDate', 'LandValue', 'DwellingValue', 'TotalValue', 'LandSqFt', 'ProvalAsmtNhood', 'TaxExemptCode', 'PropertyClassID', 'PropertyClass', 'LandUse', 'Mailable', 'MaskedOwner', 'OBJECTID', 'GlobalID', 'City', 'State', 'Shape__Area', 'Shape__Length', 'geometry']
CRS: EPSG:4326


In [3]:
# Value validation
print("Value column statistics:")
print(gdf[['LandValue', 'DwellingValue', 'TotalValue']].describe())
print(f"\nParcels with $0 land value:    {(gdf['LandValue'].fillna(0) == 0).sum():,}")
print(f"Parcels with $0 improvement:   {(gdf['DwellingValue'].fillna(0) == 0).sum():,}")
print(f"Parcels with null TaxExemptCode: {gdf['TaxExemptCode'].isna().sum():,} (taxable)")
print(f"Parcels with TaxExemptCode set:  {gdf['TaxExemptCode'].notna().sum():,} (exempt)")

Value column statistics:
          LandValue  DwellingValue    TotalValue
count  7.658800e+04   6.997700e+04  7.666900e+04
mean   1.940710e+05   5.837405e+05  7.266257e+05
std    7.266145e+05   4.215363e+06  4.482626e+06
min    1.000000e+03   1.000000e+03  1.000000e+03
25%    5.500000e+04   1.570000e+05  2.020000e+05
50%    9.000000e+04   2.510000e+05  3.280000e+05
75%    1.640000e+05   3.930000e+05  5.350000e+05
max    8.480700e+07   3.825480e+08  4.673550e+08

Parcels with $0 land value:    291
Parcels with $0 improvement:   6,902
Parcels with null TaxExemptCode: 72,456 (taxable)
Parcels with TaxExemptCode set:  4,423 (exempt)


## Section 3 — Classify and Validate

In [4]:
# Exemption flag: any parcel with a TaxExemptCode is fully exempt from local real estate tax
# Virginia exemptions governed by Va. Code § 58.1-3600 et seq. (religious, educational,
# charitable, governmental, etc.). Common city codes: 400=religious, 100=government,
# 500=educational, 711=public service corp (state-assessed), 620/630=charitable, etc.
gdf['full_exmp'] = gdf['TaxExemptCode'].notna().astype(int)
print(f"Exempt parcels: {gdf['full_exmp'].sum():,}")
print(f"Taxable parcels: {(gdf['full_exmp'] == 0).sum():,}")
print(f"\nTop TaxExemptCode values:")
print(gdf[gdf['full_exmp'] == 1]['TaxExemptCode'].value_counts().head(15))

Exempt parcels: 4,423
Taxable parcels: 72,456

Top TaxExemptCode values:
TaxExemptCode
400    591
100    523
699    452
620    411
711    387
106    377
209    364
150    303
500    184
950    169
630    140
600    129
105     66
220     46
605     45
Name: count, dtype: int64


In [5]:
# Property category mapping using Richmond's 3-digit PropertyClassID codes
# 'R' prefix = Residential assessor codes; 'B' prefix = Business; 'P' = Public
CATEGORY_MAP = {
    # Single Family Residential
    '109': 'Single Family Residential',  # R Single Family Shell
    '110': 'Single Family Residential',  # R One Story
    '115': 'Single Family Residential',  # R One Story+ (1.25, 1.5, 1.75)
    '120': 'Single Family Residential',  # R Two Story
    '130': 'Single Family Residential',  # R Two Story+ (2.5, 3.0, 3+)
    '150': 'Single Family Residential',  # R SplitLevel (Tri,Quad,Split)
    '155': 'Single Family Residential',  # R Single Family Converted ELU
    '198': 'Single Family Residential',  # R Single Family Miscellaneous
    # Small Multi-Family (2-4 units)
    '160': 'Small Multi-Family (2-4 units)',  # R Two Family Blt-As
    '161': 'Small Multi-Family (2-4 units)',  # R Two Family Converted
    '170': 'Small Multi-Family (2-4 units)',  # R Three Family Blt-As
    '171': 'Small Multi-Family (2-4 units)',  # R Three Family Converted
    '180': 'Small Multi-Family (2-4 units)',  # R Four Family Blt-As
    '181': 'Small Multi-Family (2-4 units)',  # R Four Family Converted
    '196': 'Small Multi-Family (2-4 units)',  # R LIHTC 1-4 Units Restricted
    # Large Multi-Family (5+ units)
    '309': 'Large Multi-Family (5+ units)',  # R Apartment Shell
    '310': 'Large Multi-Family (5+ units)',  # R Apartment 5-11 Units
    '315': 'Large Multi-Family (5+ units)',  # R Apartment 12-24 Units
    '321': 'Large Multi-Family (5+ units)',  # R Apartment 25-99 Units
    '325': 'Large Multi-Family (5+ units)',  # R Apartments 100+ Units
    '330': 'Large Multi-Family (5+ units)',  # R Apartments High Rise
    '345': 'Large Multi-Family (5+ units)',  # R Apartments Leasehold
    '354': 'Large Multi-Family (5+ units)',  # R Mobile Home Park/Court
    '355': 'Large Multi-Family (5+ units)',  # R Assisted Living (Licensed)
    '360': 'Large Multi-Family (5+ units)',  # R Rm Hse/Grp Home (Unlicensed)
    '396': 'Large Multi-Family (5+ units)',  # R LIHTC 5+ Units Restricted
    '398': 'Large Multi-Family (5+ units)',  # R Multi-Family Miscellaneous
    # Condominium
    '210': 'Condominium',  # R Condo Residential <11 Units
    '211': 'Condominium',  # R Condo Residential 12-49 Unit
    '212': 'Condominium',  # R Condo Residential 50+ Units
    '220': 'Condominium',  # R Condo Parking Space
    '230': 'Condominium',  # R Condo Parking Garage/Storage
    # Mixed Use
    '195': 'Mixed Use',  # R Res/Comm Mixed Use
    '335': 'Mixed Use',  # R Apt/Comm Mixed Use 5-49 Unit
    '341': 'Mixed Use',  # R Apt/Comm Mixed Use 50+ Units
    '450': 'Mixed Use',  # B Mixed Use
    # Retail / General Commercial
    '410': 'Retail / General Commercial',  # B General Retail/Service
    '415': 'Retail / General Commercial',  # B Convenience Store
    '416': 'Retail / General Commercial',  # B Supermarkets
    '418': 'Retail / General Commercial',  # B Big Box Retail
    '419': 'Retail / General Commercial',  # B Drug Stores/Pharmacy
    '420': 'Retail / General Commercial',  # B Retail Strip
    '421': 'Retail / General Commercial',  # B Neighborhood Shopping Center
    '422': 'Retail / General Commercial',  # B Community Shopping Center
    '425': 'Retail / General Commercial',  # B Regional Shopping Mall
    '426': 'Retail / General Commercial',  # B Restaurant/Bar
    '428': 'Retail / General Commercial',  # B Fast Food Restaurant
    # Office / Commercial Condo
    '440': 'Office / Commercial Condo',  # B Class A or B Office Bldg
    '441': 'Office / Commercial Condo',  # B General Office
    '442': 'Office / Commercial Condo',  # B Professional Office
    '443': 'Office / Commercial Condo',  # B Medical Clinic/Office
    '470': 'Office / Commercial Condo',  # B Commercial Condo
    '472': 'Office / Commercial Condo',  # B Commercial Common Area Unit
    '516': 'Office / Commercial Condo',  # B Research and Development
    # Hotel
    '365': 'Hotel',  # R Bed and Breakfast
    '460': 'Hotel',  # B Hotel
    '461': 'Hotel',  # B Motel
    # Industrial
    '509': 'Industrial',  # B Industrial Shell
    '510': 'Industrial',  # B Heavy Industrial
    '511': 'Industrial',  # B Light Industrial
    '532': 'Industrial',  # B Distribution Warehouse
    '533': 'Industrial',  # B Storage Warehouse
    '534': 'Industrial',  # B Transit Warehouse
    '535': 'Industrial',  # B Mini Warehouse
    '536': 'Industrial',  # B Industrial Flex Building
    '570': 'Industrial',  # B Tank Farm
    '580': 'Industrial',  # B Industrial Mines & Quarries
    '598': 'Industrial',  # B Industrial Miscellaneous
    '711': 'Industrial',  # P Public Service Corp. (utilities — state-assessed)
    '715': 'Industrial',  # B Railroad Rail/Imp
    # Transportation - Parking
    '106': 'Transportation - Parking',  # R Paved Parking
    '306': 'Transportation - Parking',  # R Apartment Parking Lot/Deck
    '406': 'Transportation - Parking',  # B Paved Surface Parking
    '490': 'Transportation - Parking',  # B Parking Deck
    '506': 'Transportation - Parking',  # B Industrial Paved Parking
    # Vacant Land
    '101': 'Vacant Land',  # R Single Family Vacant (R1-R7)
    '191': 'Vacant Land',  # R Site Improvements
    '201': 'Vacant Land',  # R Condo Vacant Land
    '301': 'Vacant Land',  # R Multi-Family Vacant (R43&R48)
    '302': 'Vacant Land',  # R Multi-Family Vacant (R53)
    '303': 'Vacant Land',  # R Multi-Family Vacant (R73)
    '401': 'Vacant Land',  # B Commercial Vacant Land
    '501': 'Vacant Land',  # B Industrial Vacant Land
    # Exempt / Institutional (many will also have TaxExemptCode set)
    '102': 'Exempt',  # R Island (public waterway land)
    '105': 'Exempt',  # R Park/Playground/Common Area
    '205': 'Exempt',  # R Condo Common Area Main
    '206': 'Exempt',  # R Condo Common Area Unit
    '305': 'Exempt',  # B Multi-Family Common Area
    '405': 'Exempt',  # B Commercial Common Area
    '452': 'Exempt',  # B Sports Complex/Convention Ctr
    '455': 'Exempt',  # B Community Ctr / Club
    '456': 'Exempt',  # B Educational
    '457': 'Exempt',  # B University
    '458': 'Exempt',  # B Library/Museum/Capitol/Court
    '466': 'Exempt',  # B Fire/Police/Public
    '467': 'Exempt',  # B Hospital/Nursing Homes
    '468': 'Exempt',  # B Religious/Church/Synagogue
    '469': 'Exempt',  # B Cemetery
    '471': 'Exempt',  # B Commercial Common Area Main
    '491': 'Exempt',  # B Air Space
    # Other Commercial
    '409': 'Other Commercial',  # B Commercial Shell
    '430': 'Other Commercial',  # B Vehicle Sales & Service
    '432': 'Other Commercial',  # B Vehicle Service/Car Wash
    '445': 'Other Commercial',  # B Bank
    '446': 'Other Commercial',  # B Theater
    '447': 'Other Commercial',  # B Health/Exercise Facility
    '448': 'Other Commercial',  # B Day Care Facility
    '449': 'Other Commercial',  # B Funeral Home
    '454': 'Other Commercial',  # B City/Country Club
    '480': 'Other Commercial',  # B Commercial Leasehold
    '482': 'Other Commercial',  # B Towers & Equipment
    '498': 'Other Commercial',  # B Commercial Miscellaneous
    # Other / miscellaneous
    '190': 'Other',  # R Garage/Outbuilding (standalone structure, no associated parcel)
    '193': 'Other',  # R Residential Leasehold
    '750': 'Other',  # R Ground Lease - MWCLT (Maggie Walker Community Land Trust)
}

gdf['PROPERTY_CATEGORY'] = gdf['PropertyClassID'].map(CATEGORY_MAP).fillna('Other')

# Override: $0 improvement value → Vacant Land (catches miscoded vacant parcels)
gdf.loc[gdf['DwellingValue'].fillna(0) <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override: exempt flag wins over the land-use category
gdf.loc[gdf['full_exmp'] == 1, 'PROPERTY_CATEGORY'] = 'Exempt'

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nNull PROPERTY_CATEGORY: {gdf['PROPERTY_CATEGORY'].isna().sum()}")

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential         52745
Vacant Land                        5275
Exempt                             4539
Condominium                        4223
Small Multi-Family (2-4 units)     3658
Retail / General Commercial        1137
Large Multi-Family (5+ units)      1065
Office / Commercial Condo           858
Mixed Use                           818
Industrial                          803
Transportation - Parking            713
Other Commercial                    608
Other                               384
Hotel                                53
Name: count, dtype: int64

Null PROPERTY_CATEGORY: 0


In [6]:
# Inspect residual 'Other' parcels to confirm they are not large policy-relevant categories
other_mask = gdf['PROPERTY_CATEGORY'] == 'Other'
print(f"'Other' parcels: {other_mask.sum():,}")
print(gdf[other_mask]['PropertyClass'].value_counts())

'Other' parcels: 384
PropertyClass
R Garage/Outbuilding       311
R Ground Lease - MWCLT      72
R Residential Leasehold      1
Name: count, dtype: int64


## Section 4 — Current Tax Model

**Virginia assessment system**: Full market value (100%) — no assessment ratio  
**City rate**: $1.20 per $100 = 12.0 mills per $1,000  
**Exemptions**: TaxExemptCode IS NOT NULL → excluded entirely (current_tax = 0)

In [7]:
# Taxable values: fill null DwellingValue (vacant parcels) with 0
gdf['taxable_land_value'] = gdf['LandValue'].fillna(0).clip(lower=0)
gdf['taxable_improvement_value'] = gdf['DwellingValue'].fillna(0).clip(lower=0)
gdf['taxable_total_value'] = gdf['TotalValue'].fillna(0).clip(lower=0)

# Consistency check: taxable_total should equal land + improvement
gap = (gdf['taxable_land_value'] + gdf['taxable_improvement_value'] - gdf['taxable_total_value']).abs()
print(f"Land + Improvement vs Total mismatch > $1: {(gap > 1).sum():,} parcels")
# If mismatch is large, use land + improvement as the base for modeling
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"\nModeled revenue:  ${current_revenue:,.0f}")
print(f"Official FY2025:  ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:              {gap_pct:+.2f}%")

Land + Improvement vs Total mismatch > $1: 2 parcels

Modeled revenue:  $525,987,000
Official FY2025:  $494,000,000
Gap:              +6.48%


## Section 5 — Split-Rate Model (4:1)

In [8]:
# Exclude fully-exempt parcels from the reform
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"Revenue check:       ${new_revenue:,.0f} (target: ${taxable['current_tax'].sum():,.0f})")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Richmond, VA — 4:1 Split-Rate Tax Impact")

Land millage:        26.2905 mills


Improvement millage: 6.5726 mills
Revenue check:       $525,987,000 (target: $525,987,000)


Richmond, VA — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential  52745        $919,333        0.3%        $17         $-17    2.0%      -0.5%            28.4%            23.9%
                   Vacant Land   5275      $7,918,636      115.7%     $1,501         $743  114.2%     119.1%            96.0%             0.0%
                        Exempt   4539       $-516,789      -11.9%      $-114           $0    0.5%       0.0%             1.3%             0.7%
                   Condominium   4223     $-2,197,434      -15.4%      $-520        $-475  -14.6%     -16.6%             3.1%            78.3%
Small Multi-Family (2-4 units)   3658      $1,454,345        6.3%       $398         $282    5.2%       5.5%            41.0%            23.3%
   Reta

## Section 6 — Exploration Charts

In [9]:
# Quick bar chart of median tax change % by category
taxable_summary = gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt'].copy()
summary = (
    taxable_summary
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in summary.values]
summary.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Richmond, VA — Median Tax Change % by Category (4:1 Split-Rate)', fontsize=12)
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("Saved category_preview.png")

Saved category_preview.png


## Section 7 — Census Join + Export

In [10]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [11]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/richmond/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ richmond: 76,879 rows → ../../analysis/data/richmond.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | +6.48% vs official ~$494M (FY2025 budget projection). Gap is consistent with Richmond's known ~$31M in delinquent real estate taxes (~6% non-collection rate). Theoretical gross levy = ~$526M; official figure reflects net expected collections. |
| Parcel count | 76,879 city parcels downloaded (Richmond is an independent city; no filter needed) |
| Census coverage | 100.0% matched to block groups |
| PNGs generated | 7 of 7 |
| SFR median change | -0.5% (slight decrease — households with improvement-heavy parcels benefit) |
| Vacant land median change | +119.1% (large increase — correct direction for LVT) |
| Parking median change | +114.4% (large increase — correct direction) |

**Revenue gap note**: Richmond has a known $31M+ in uncollected delinquent real estate taxes (reported FY2025). The theoretical gross levy at $1.20/$100 is ~$526M; the ~$494M official figure represents net budgeted collections after expected non-payment. A +6.5% gap between theoretical and budgeted is consistent with this delinquency. The model correctly computes the theoretical levy.

**MWCLT note**: 72 parcels classified as 'Ground Lease - MWCLT' (Maggie Walker Community Land Trust) fall in the 'Other' category. These are community land trust parcels where the trust holds the land and residents own the structure. Under LVT, the land-owning trust would bear higher land tax burden — a policy-relevant consideration not further modeled here.